# TD 4 : réseaux de neurones, et ce qu'ils coûtent

## Comment se passe cette journée

Vous travaillez **en binôme**, et vous changez de clavier à chaque étape. Celui qui ne tape pas lit l'énoncé à voix haute et vérifie.

**Personne ne lève la main pour savoir s'il a juste.** Chaque exercice se termine par une cellule de vérification qui vous le dit, et qui explique ce qui ne va pas si ce n'est pas bon. Les indices sont repliés sous chaque exercice : ouvrez-les plutôt que d'attendre.

**Quand vous finissez une étape**, marquez-la sur le tableau au mur. C'est ce qui permet de voir où en est la salle sans faire le tour des rangs.

## Ce que vous saurez faire

1. écrire les cinq lignes d'une boucle d'entraînement PyTorch
2. vérifier à la main que la chain rule donne bien ce que calcule autograd
3. casser un réseau de trois façons différentes et reconnaître chaque panne
4. comparer un réseau et un arbre sur autre chose que la performance

## Prérequis

- avoir fini le TD 3
- avoir suivi le bloc D de la journée 1

**Durée :** 70 minutes

## Le contexte

Mêmes données qu'au TD 3, même question, même split. On remplace seulement le decision tree par un neural network, et on regarde ce que cela change.

## Le déroulé

| quand | quoi |
|---|---|
| 5:45 | étape 1 : les cinq lignes |
| 6:05 | étape 2 : vérifier la chain rule à la main |
| 6:25 | étape 3 : trois façons de le casser |
| 6:45 | étape 4 : le verdict, en euros |
| 7:00 | TD 5 |

In [ ]:
# Amorce : a executer en premier, une seule fois par notebook.
import sys, pathlib
racine = pathlib.Path.cwd()
while not (racine / "aeolis").is_dir() and racine != racine.parent:
    racine = racine.parent
sys.path.insert(0, str(racine))
import os
os.chdir(racine)

import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aeolis import plots, evaluate as ev

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

# Les donnees sont dans cours_fr/data pendant le cours, et dans donnees/ dans le
# dossier qui vous est remis. On prend celui qui existe, pour que le meme notebook
# fonctionne dans les deux cas.
DONNEES = next((c for c in (racine / "cours_fr" / "data", racine / "donnees",
                            racine / "data") if c.is_dir()), racine)

print("racine du cours :", racine.name)
print("donnees :", DONNEES.name, "->",
      sorted(p.name for p in DONNEES.glob("*.parquet")) or "aucun parquet")
print("pret")

## Préparation : on reprend le jeu du TD 3

Cette cellule refait le travail du TD 3 en une fois. Exécutez-la et passez à la suite : il n'y a rien de nouveau ici.

In [ ]:
# Fourni, rien a faire : on reconstruit le jeu du TD 3.
import torch
import torch.nn as nn
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sejours = pd.read_parquet(DONNEES / 'readmission.parquet')
exclus = ['11', '19', '20', '21', '13', '14']
vivants = sejours[~sejours['discharge_disposition_id'].isin(exclus)].copy()
y = (vivants['readmitted'] == '<30').astype(int).values

NUM = ['time_in_hospital', 'num_lab_procedures', 'num_procedures',
       'num_medications', 'number_outpatient', 'number_emergency',
       'number_inpatient', 'number_diagnoses']
CAT = ['race', 'gender', 'age', 'admission_type_id', 'admission_source_id',
       'A1Cresult', 'max_glu_serum', 'change', 'diabetesMed',
       'insulin', 'metformin']

X = vivants[NUM + CAT].copy()
for c in NUM:
    X[c] = pd.to_numeric(X[c])
for c in CAT:
    X[c] = X[c].replace('?', 'Inconnu').fillna('Inconnu').astype(str)
    X[c] = X[c].astype('category')

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

# Un reseau ne mange pas de categories : il faut tout encoder en nombres,
# et mettre les numeriques a la meme echelle. L arbre n avait besoin de rien.
prep = ColumnTransformer([
    ('num', StandardScaler(), NUM),
    ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=30,
                          sparse_output=False), CAT)])
Xtr = prep.fit_transform(X_tr).astype('float32')
Xte = prep.transform(X_te).astype('float32')

print('apres encodage :', Xtr.shape[1], 'colonnes')
print('train', Xtr.shape, '  test', Xte.shape)

---

## Étape 1 : Les cinq lignes

**20 minutes.** Toute boucle d'entraînement PyTorch jamais écrite tient dans ces cinq lignes. Vous allez les remplir.

## Mémo de code

Chaque exercice ci-dessous se résout en adaptant un de ces patrons. Copiez, changez les noms, exécutez. Ce n'est pas tricher : c'est ainsi que l'on écrit du code de data.

```python
# un reseau : des couches empilees
reseau = nn.Sequential(nn.Linear(entrees, 64), nn.ReLU(),
                       nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))

# la loss des problemes oui/non
perte = nn.BCEWithLogitsLoss()

# l optimiseur
optimiseur = torch.optim.Adam(reseau.parameters(), lr=1e-3)

# les donnees, en tenseurs
Xt = torch.tensor(Xtr)
y_t = torch.tensor(y_tr, dtype=torch.float32).unsqueeze(1)

# predire sans suivre les gradients
with torch.no_grad():
    proba = torch.sigmoid(reseau(Xte_t)).numpy().ravel()
```

### 1.1 Écrivez la boucle

Cinq blancs, un par ligne. Les commentaires disent lequel est lequel.

In [ ]:
torch.manual_seed(42)
reseau = nn.Sequential(nn.Linear(Xtr.shape[1], 64), nn.ReLU(),
                       nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))
perte = nn.BCEWithLogitsLoss()
optimiseur = torch.optim.Adam(reseau.parameters(), lr=1e-3)

Xtr_t = torch.tensor(Xtr)
ytr_t = torch.tensor(y_tr, dtype=torch.float32).unsqueeze(1)

for epoque in range(60):
    ...                          # (1) oublier les gradients du pas precedent
    sortie = ...                 # (2) forward pass
    erreur = ...                 # (3) a quel point on se trompe
    ...                          # (4) un gradient pour CHAQUE poids
    ...                          # (5) un pas vers le bas

print(f'loss finale : {erreur.item():.4f}')

**Indice 1.** Les cinq noms sont `optimiseur.zero_grad()`, `reseau(Xtr_t)`, `perte(sortie, ytr_t)`, `erreur.backward()`, `optimiseur.step()`.

**Indice 2.** Dans cet ordre exactement. Le piège classique est d'oublier le (5), auquel cas la loss ne bouge pas d'une époque à l'autre.

In [ ]:
poids = sum(p.numel() for p in reseau.parameters())
assert erreur.item() < 0.40, (
    f'loss finale {erreur.item():.4f}, attendue vers 0,34. '
    'Si elle ne bouge pas, optimiseur.step() manque.')
Xte_t = torch.tensor(Xte)
with torch.no_grad():
    proba_reseau = torch.sigmoid(reseau(Xte_t)).numpy().ravel()
auc_reseau = roc_auc_score(y_te, proba_reseau)
assert auc_reseau > 0.60, f'AUC {auc_reseau:.4f}, attendue vers 0,63'
print(f'correct. {poids:,} poids appris, loss {erreur.item():.4f}, '
      f'AUC {auc_reseau:.4f}.')
print('Souvenez-vous de ce chiffre : l arbre du TD 3 faisait 0,6397.')

---

## Étape 2 : Vérifier la chain rule à la main

**20 minutes.** autograd applique la chain rule. Vous allez le vérifier au lieu de le croire.

On prend le plus petit réseau qui ait plus d'un chemin : **deux entrées, deux unités cachées, une sortie**. Les poids sont posés à la main pour que la figure soit la même pour tout le monde et qu'aucune unité ne soit morte.

In [ ]:
# Fourni : un reseau minuscule, un forward, un backward.
petit = nn.Sequential(nn.Linear(2, 2), nn.ReLU(), nn.Linear(2, 1))
with torch.no_grad():
    petit[0].weight.copy_(torch.tensor([[0.6, -0.5], [0.4, 0.9]]))
    petit[0].bias.copy_(torch.tensor([0.1, 0.2]))
    petit[2].weight.copy_(torch.tensor([[0.7, -1.1]]))
    petit[2].bias.copy_(torch.tensor([0.05]))

x_un = torch.tensor([[1.2, -0.4]])
y_un = torch.tensor([[1.0]])
cache = petit[1](petit[0](x_un))
prediction = petit[2](cache)
loss_petit = ((prediction - y_un) ** 2).mean()
loss_petit.backward()

print('activations cachees :', cache.detach().numpy().ravel())
print(f'prediction {prediction.item():.4f}, loss {loss_petit.item():.4f}')
print('gradients de la premiere couche :')
print(petit[0].weight.grad.numpy())

### 2.1 Multipliez la chain rule vous-même

Suivez le poids qui va de `x1` vers `h1`. Le bouger change `h1`, qui change la prédiction, qui change la loss. **La chain rule dit de multiplier ces effets.** Le chemin est court, faites-le à la main.

In [ ]:
# A FAIRE : les quatre facteurs, puis leur produit.
d_loss_d_prediction = ...      # la loss est (prediction - y) au carre
d_prediction_d_h1 = ...        # le poids suivant sur le chemin
d_h1_d_preactivation = 1.0     # ReLU, et h1 est positif
d_preactivation_d_poids = ...  # l entree de ce poids

a_la_main = (d_loss_d_prediction * d_prediction_d_h1
             * d_h1_d_preactivation * d_preactivation_d_poids)
autograd = petit[0].weight.grad[0, 0].item()

print(f'chain rule multipliee a la main : {a_la_main:.6f}')
print(f'ce que loss.backward() a calcule : {autograd:.6f}')

**Indice 1.** La dérivée de `(p - y)**2` par rapport à `p` vaut `2 * (p - y)`. Le poids suivant est `petit[2].weight[0, 0]`. L'entrée de ce poids est `x_un[0, 0]`.

**Indice 2.** `d_loss_d_prediction = 2 * (prediction.item() - y_un.item())`, `d_prediction_d_h1 = petit[2].weight[0, 0].item()`, `d_preactivation_d_poids = x_un[0, 0].item()`

In [ ]:
# 1e-6 et non zero : tout est en float32 ici, et les deux chemins de calcul
# arrondissent a des endroits differents. Le residu est de l arithmetique,
# pas une difference de mathematiques.
assert abs(a_la_main - autograd) < 1e-6, (
    f'les deux doivent coincider : {a_la_main:.6f} contre {autograd:.6f}')
print('Les deux donnent -0,987840.')
print()
print('autograd ne fait pas autre chose que ce que vous venez de faire :')
print('il le fait 6 337 fois, 60 fois de suite, sans se tromper.')
print()
print('Et la tolerance n est pas zero. Les deux nombres different a la')
print('neuvieme decimale, parce que tout est en float32. C est le meme')
print('bruit qui fait qu un reseau entraine deux fois ne donne pas deux')
print('fois le meme resultat.')

---

## Étape 3 : Trois façons de le casser

**15 minutes.** Un arbre n'a aucun de ces réglages. C'est le vrai argument.

In [ ]:
# A FAIRE : relancer l entrainement avec trois reglages fautifs,
#           et noter l AUC et la loss finale de chacun.
def essayer(lr, donnees_tr=None, donnees_te=None, etiquette=''):
    a = Xtr if donnees_tr is None else donnees_tr
    b = Xte if donnees_te is None else donnees_te
    torch.manual_seed(42)
    net = nn.Sequential(nn.Linear(a.shape[1], 64), nn.ReLU(),
                        nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    at = torch.tensor(np.asarray(a, dtype='float32'))
    bt = torch.tensor(np.asarray(b, dtype='float32'))
    for _ in range(60):
        ...                      # les cinq lignes, encore
    with torch.no_grad():
        p = torch.sigmoid(net(bt)).numpy().ravel()
    auc = roc_auc_score(y_te, p)
    print(f'{etiquette:<34} AUC {auc:.4f}   loss {e.item():.4f}')
    return auc

auc_bon = essayer(1e-3, etiquette='learning rate 1e-3 (correct)')
auc_grand = essayer(1.0, etiquette='learning rate 1,0 (trop grand)')
auc_petit = essayer(1e-6, etiquette='learning rate 1e-6 (trop petit)')

**Indice 1.** Le corps de la boucle est exactement celui de l'étape 1 : zero_grad, forward, loss, backward, step. Nommez l'erreur `e` pour que le `print` fonctionne.

**Indice 2.** `opt.zero_grad()` puis `e = perte(net(at), ytr_t)` puis `e.backward()` puis `opt.step()`

In [ ]:
assert abs(auc_grand - 0.5000) < 0.02, (
    f'avec lr=1,0 l AUC devrait tomber a 0,50, obtenu {auc_grand:.4f}')
assert auc_petit < 0.55, (
    f'avec lr=1e-6 l AUC devrait etre mauvaise, obtenu {auc_petit:.4f}')
print()
print(f'lr trop grand : AUC {auc_grand:.4f}. Exactement le hasard.')
print('   Le reseau a diverge et n a rien appris du tout.')
print(f'lr trop petit : AUC {auc_petit:.4f}. PIRE que le hasard.')
print('   Et c est la loss finale qui le signale, pas l AUC.')
print()
print('Un reseau peut echouer en silence. Un arbre n a pas de learning rate.')

### 3.1 Et sans scaling

In [ ]:
# A FAIRE : refaire tourner avec les colonnes numeriques NON mises a l echelle.
brut_tr = np.column_stack([X_tr[NUM].values.astype('float32'), Xtr[:, len(NUM):]])
brut_te = np.column_stack([X_te[NUM].values.astype('float32'), Xte[:, len(NUM):]])

auc_sans_scaling = ...

**Indice 1.** Réutilisez `essayer`, avec `donnees_tr=brut_tr` et `donnees_te=brut_te`, et un learning rate correct.

**Indice 2.** `auc_sans_scaling = essayer(1e-3, brut_tr, brut_te, 'sans scaling')`

In [ ]:
assert auc_sans_scaling < auc_bon, (
    'sans scaling, le reseau devrait etre moins bon')
print(f'perte due au seul oubli du scaling : '
      f'{auc_bon - auc_sans_scaling:.4f} point d AUC.')
print('Une ligne oubliee. Le tree, lui, se moque completement de l echelle.')

---

## Étape 4 : Le verdict, en euros

**15 minutes.** Comparer deux modèles sur autre chose que leur performance.

In [ ]:
# A FAIRE : le benefice net du programme, pour le reseau et pour l arbre.
#           Reprenez la fonction du TD 3.
from sklearn.ensemble import HistGradientBoostingClassifier
COUT_READMISSION, COUT_PROGRAMME, EFFICACITE = 8000.0, 500.0, 0.20

arbre = HistGradientBoostingClassifier(
    random_state=42, categorical_features=CAT).fit(X_tr, y_tr)
proba_arbre = arbre.predict_proba(X_te)[:, 1]

def meilleur_benefice(proba):
    ...

b_arbre = meilleur_benefice(proba_arbre)
b_reseau = meilleur_benefice(proba_reseau)

print(f"{'':<22}{'AUC':>9}{'benefice':>14}")
print(f"{'arbre':<22}{roc_auc_score(y_te, proba_arbre):>9.4f}{b_arbre:>14,.0f}")
print(f"{'reseau':<22}{roc_auc_score(y_te, proba_reseau):>9.4f}{b_reseau:>14,.0f}")

**Indice 1.** C'est exactement la fonction `benefice` du TD 3, mais balayée sur une grille de seuils dont on garde le maximum.

**Indice 2.** Boucle sur `np.linspace(0.01, 0.99, 99)`, calcule `EFFICACITE * y_te[inscrits].sum() * COUT_READMISSION - inscrits.sum() * COUT_PROGRAMME`, renvoie le max.

In [ ]:
ecart_auc = roc_auc_score(y_te, proba_arbre) - roc_auc_score(y_te, proba_reseau)
assert b_arbre > b_reseau, 'sur ce probleme, l arbre doit gagner'
print(f'ecart d AUC      : {ecart_auc:+.4f}, ce qui parait negligeable')
print(f'ecart de benefice : {b_arbre - b_reseau:+,.0f} EUR, ce qui ne l est pas')
print()
print('Un ecart de metrique ne veut rien dire tant qu il n est pas traduit')
print('dans l unite du metier. Ici, un centieme d AUC change qui entre dans')
print('le programme, donc combien de readmissions sont evitees.')

## Ce que cette étape a établi

- Cinq lignes : zero_grad, forward, loss, backward, step. Oublier la cinquième laisse la loss immobile.
- La chain rule multipliée à la main donne -0,987840, exactement ce que calcule autograd. Tolérance 1e-6 et non zéro, parce que float32.
- Learning rate trop grand : AUC 0,50, le hasard. Trop petit : pire que le hasard, et seule la loss le signale.
- Oublier le scaling coûte des points d'AUC pour une ligne manquante.
- L'arbre gagne sur l'AUC et sur le bénéfice, et n'a aucun de ces quatre réglages.

## Les erreurs classiques

- **Oublier `optimiseur.zero_grad()`.** Les gradients s'accumulent d'un pas sur l'autre et l'entraînement part en vrille.
- **Comparer deux modèles sur l'AUC seule.** Traduisez l'écart dans l'unité du métier avant de conclure.
- **Conclure qu'un réseau est toujours meilleur.** Sur des données en tableau, c'est l'inverse, et vous venez de le mesurer.

## Pour aller plus loin, si vous avez fini en avance

- Relancer l'étape 1 avec 300 époques au lieu de 60. L'AUC de test s'améliore-t-elle, ou le réseau commence-t-il à apprendre le bruit ?
- Ajouter une couche cachée de plus. L'AUC bouge-t-elle assez pour justifier le temps passé ?